# 🚀 Günlük Bülten — Colab GPU İşçisi

Panel **"Sıra Colab'da"** dediğinde buraya gel ve **tek hücreyi** çalıştır (▶). Hücre şunları kendiliğinden yapar:

1. Drive'ı bağlar, kodu indirir, gizli anahtarları Colab Secrets'tan okur.
2. Veritabanından bekleyen günü bulur; Colab'a iş yoksa hiçbir şey kurmadan çıkar.
3. **Görselleri** (FLUX) üretir, ardından **4 dil klon sesi** (Chatterbox) üretir.
4. Bitince panel kendiliğinden **"Sıra sende: TR ses kaydı"** der.

### Bir kerelik kurulum
- **Runtime → Change runtime type → T4 GPU** (TPU değil).
- Soldaki 🔑 **Secrets** paneline şunları ekle ve *Notebook access* anahtarını aç:
  `SUPABASE_URL`, `SUPABASE_SERVICE_KEY` (zorunlu).
  `HF_TOKEN` (yerel FLUX için zorunlu): önce [FLUX.1-schnell sayfasında](https://huggingface.co/black-forest-labs/FLUX.1-schnell) *Agree and access repository* de, sonra [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) → *Read* token.
- Drive'da `MyDrive/gunluk-bulten/refs/ref_voice.wav` dosyası olsun (10–20 sn temiz konuşma kaydın).
- PC'deki `config.yaml` dosyasını istersen `MyDrive/gunluk-bulten/config.yaml` olarak koy; varsa o kullanılır, yoksa repodaki.

In [ ]:
# ▶ TEK HÜCRE — Colab GPU işçisi
import os, shutil, subprocess, sys
from google.colab import drive, userdata

REPO_URL = 'https://github.com/Oguzhankuri/RSSBot.git'
DRIVE_ROOT = '/content/drive/MyDrive/gunluk-bulten'
CODE_DIR = '/content/RSSBot'

drive.mount('/content/drive')
os.makedirs(f'{DRIVE_ROOT}/output', exist_ok=True)

def sh(cmd):
    print('$', cmd)
    return subprocess.run(cmd, shell=True).returncode

if os.path.exists(CODE_DIR):
    sh(f'git -C {CODE_DIR} pull --ff-only')
else:
    sh(f'git clone --depth 1 {REPO_URL} {CODE_DIR}')
os.chdir(CODE_DIR)

# Drive'daki config ve referans ses (varsa) koda kopyalanır
if os.path.exists(f'{DRIVE_ROOT}/config.yaml'):
    shutil.copy(f'{DRIVE_ROOT}/config.yaml', 'config.yaml')
os.makedirs('refs', exist_ok=True)
if os.path.exists(f'{DRIVE_ROOT}/refs/ref_voice.wav'):
    shutil.copy(f'{DRIVE_ROOT}/refs/ref_voice.wav', 'refs/ref_voice.wav')

def secret(name, required=True):
    try:
        return userdata.get(name)
    except Exception:
        if required:
            raise SystemExit(f'❌ Colab Secrets içinde {name} yok. Soldaki 🔑 panelinden ekle.')
        return ''

with open('.env', 'w') as fh:
    fh.write('\n'.join([
        f"SUPABASE_URL={secret('SUPABASE_URL')}",
        f"SUPABASE_SERVICE_KEY={secret('SUPABASE_SERVICE_KEY')}",
        f"HF_TOKEN={secret('HF_TOKEN', required=False)}",
        f'BULTEN_OUTPUT_DIR={DRIVE_ROOT}/output',
        'BULTEN_DB_PROVIDER=supabase',
        '',
    ]))

sh('pip install -q -r requirements.txt')
PIPE = f'{sys.executable} -m bulten.pipeline'
if sh(f'{PIPE} colab-turn --date latest') != 0:
    print('\n✋ Şu an Colab\'a iş yok. Panelde sırayı kontrol et.')
else:
    import yaml
    provider = yaml.safe_load(open('config.yaml', encoding='utf-8'))['images']['provider']
    if provider == 'flux_local':
        sh('pip install -q -r requirements-gpu.txt')
    if sh(f'{PIPE} gpu --only images --date latest') == 0:
        # Chatterbox kendi torch/diffusers sürümlerini kurar → görsellerden SONRA
        sh('pip install -q -r requirements-voice.txt')
        sh(f'{PIPE} gpu --only voices --date latest')
    sh(f'{PIPE} status --date latest')
    print('\n✅ Colab işi bitti. Panele dön.')
